# 第2章 Python 数据科学工具栈 —— 配套代码

对应正文 `book/part1/02-python-toolkit.md`。运行前请先生成内置数据：
`uv run python scripts/make_sample_data.py`。

## NumPy：向量化与广播

In [ ]:
import numpy as np

prices = np.array([10.0, 10.5, 10.3, 10.8])
returns = prices[1:] / prices[:-1] - 1   # 向量化：一次算出所有日收益
returns

In [ ]:
rng = np.random.default_rng(0)
mat = rng.standard_normal((100, 4))   # 100 天 x 4 只股票
demeaned = mat - mat.mean(axis=0)     # 广播：mean 形状 (4,) 自动对齐到 (100,4)
demeaned.mean(axis=0).round(6)        # 每列均值应≈0

## Pandas：加载内置数据与时间索引

In [ ]:
from fds import load_sample_prices

prices = load_sample_prices()
print(type(prices.index))
prices.head()

In [ ]:
# 按时间区间切片：取 2025 年第一季度
prices.loc["2025-01":"2025-03"].head()

## 重采样：日 -> 月

In [ ]:
month_end = prices.resample("ME").last()   # 每月最后一个交易日收盘价
monthly_ret = month_end.pct_change().dropna()
monthly_ret.head()

## 读写：Parquet 往返一致性

In [ ]:
import tempfile, os

tmp = os.path.join(tempfile.gettempdir(), "prices_roundtrip.parquet")
prices.to_parquet(tmp)
back = __import__("pandas").read_parquet(tmp)
print("往返一致：", prices.equals(back))

## 练习

1. 把日度价格重采样为**周度**收益率（`resample("W")`）。
2. 用 `rolling(20).mean()` 计算 `LIQUOR` 的 20 日移动平均。
3. 把波动最大的股票单独存为 Parquet 再读回，验证一致。